In [3]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END

MAX_TRIES = 3

class State(TypedDict):
    draft: str 
    score: int
    tries: int 


def write(state: State) -> dict:
    n = state['tries'] + 1
    return {"draft": f"draft v{n}", "tries": n}

def judge(state: State) -> dict:
    # stand-in for an LLM judge: scores only get good enough on try 2
    return {"score": 40 * state["tries"]}

def after_judge(state: State) -> Literal["write", "done", "escalate"]:
    if state["score"] >= 70:
        return "done"
    elif state["tries"] >= MAX_TRIES:
        return "escalate"
    else:
        return "write"
    
def done(state: State):
    return {"draft": state["draft"] + " (approved)"}

def escalate(state: State):
    return {"draft": state["draft"] + " (ESCALATED to human)"}

g = StateGraph(State)
for name, fn in [("write", write), ("judge", judge), ("done", done), ("escalate", escalate)]:
    g.add_node(name, fn)
g.add_edge(START, "write")
g.add_edge("write", "judge")
g.add_conditional_edges("judge", after_judge)
g.add_edge("done", END)
g.add_edge("escalate", END)

graph = g.compile()

In [4]:
print("normal  :", graph.invoke({"draft": "", "score": 0, "tries": 0}))

normal  : {'draft': 'draft v2 (approved)', 'score': 80, 'tries': 2}


always-failing judge: must end in escalation, never loop forever

In [5]:
import types 

def bad_judge(state): return {"score": 0}

g2 = StateGraph(State)

for name, fn in [("write", write), ("judge", bad_judge), ("done", done), ("escalate", escalate)]:
    g2.add_node(name, fn)
g2.add_edge(START, "write")
g2.add_edge("write", "judge")
g2.add_conditional_edges("judge", after_judge)
g2.add_edge("done", END)
g2.add_edge("escalate", END)

In [7]:
print("bad judge:", g2.compile().invoke({"draft": "", "score": 0, "tries": 0}))

bad judge: {'draft': 'draft v3 (ESCALATED to human)', 'score': 0, 'tries': 3}


what happens with NO bound? the runtime's recursion limit is the safety net

In [8]:
from langgraph.errors import GraphRecursionError
def forever(state: State) -> Literal["write"]: return "write"

g3 = StateGraph(State)
g3.add_node("write", write)
g3.add_node("judge", judge)
g3.add_edge(START, "write")
g3.add_edge("write", "judge")

g3.add_conditional_edges("judge", forever)


In [9]:

try:
    g3.compile().invoke({"draft": "", "score": 0, "tries": 0}, {"recursion_limit": 8})
except GraphRecursionError as e:
    print("unbounded:", type(e).__name__, "-", str(e).splitlines()[0])


unbounded: GraphRecursionError - Recursion limit of 8 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
